In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

Run this notebook with the repository root as the working directory. The settings below match `run_kernel_t2i.sh`: FLUX.1-schnell, sketch, RBF, and eta=35.0. Set `seed` to the seed passed to the launcher (42 in this example).

Generate KAS results with `bash run_kernel_t2i.sh 42 0`. For the matching Mean-AcT baseline, change `for sty in cyberpunk; do` to `for sty in sketch; do` in `run_t2i.sh`, then run `bash run_t2i.sh 42 0`. Use the same seed for both methods. Existing cyberpunk results are not a matching baseline for sketch.

These settings follow the current launcher; they do not establish exact reproduction of the paper.


In [ ]:
model = "FLUX.1-schnell"
style = "cyberpunk"  # same style as passed to run_kernel_t2i.sh and run_t2i.sh
kernel_type = "rbf"
eta = 35.0
seed = 42


In [ ]:
baseline_dir = Path(f"results_{model}_mean_ot_{style}_{seed}")
kas_dir = Path(f"results_{model}_kernel_{kernel_type}_{style}_eta_{eta}_{seed}")
baseline_csv = baseline_dir / "calculate_clip_score" / "clip_score.csv"
kas_csv = kas_dir / "calculate_clip_score" / "clip_score.csv"

missing_csvs = [path for path in (baseline_csv, kas_csv) if not path.is_file()]
if missing_csvs:
    raise FileNotFoundError(
        "Missing experiment CSVs:\n"
        + "\n".join(str(path.resolve()) for path in missing_csvs)
        + "\nGenerate both methods with the configured style and seed before loading."
    )

clip_data = pd.read_csv(baseline_csv, index_col=0)
clip_data.head()


In [ ]:
clip_data_kas = pd.read_csv(kas_csv, index_col=0)

clip_data_kas.head()


In [ ]:
fig = go.Figure()

lbs = sorted(clip_data['strength'].unique())
x_lbs = [0.05 if lb == 0 else lb for lb in lbs]

zeroshot_mean_ot = [clip_data[
    (clip_data['strength'] == lb) & (clip_data['conditional_zero_shot_score'] >= 0.49)
].shape[0] / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs]

zeroshot_kas = [clip_data_kas[
    (clip_data_kas['strength'] == lb) & (clip_data_kas['conditional_zero_shot_score'] >= 0.49)
].shape[0] / clip_data_kas[clip_data_kas['strength'] == 1].shape[0] for lb in lbs]

# --- First plot: line across all points ---
fig.add_trace(go.Scatter(
    x=x_lbs,
    y=zeroshot_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT"
))

# --- Clip Data (KAS) ---
fig.add_trace(go.Scatter(
    x=x_lbs,
    y=zeroshot_kas,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="KAS"
))

# Layout
fig.update_layout(
    paper_bgcolor="white",
    width=700,
    height=500,
    showlegend=False,
    legend=dict(font=dict(size=18)),  # legend font
    font=dict(size=18),               # default font for all text
    xaxis=dict(
        domain=[0, 1.0],
        title="Strength λ",
        type="log",
        tickvals=[0.1, 1, 10],
        ticktext=["0.1", "1", "10"],
        tickfont=dict(size=16)
    ),
    yaxis=dict(
        title="0-shot(style) (→)",
        # titlefont=dict(size=20),
        tickfont=dict(size=16)
    ),
    margin=dict(t=40, b=60, l=80, r=20)
)

fig.show()
output_path = Path(f"visualization/{model}/{style}/0shot_clip_{style}.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(output_path, width=700, height=500, scale=2)

In [ ]:
fig = go.Figure()

lbs = sorted(clip_data['strength'].unique())
x_lbs = [0.05 if lb == 0 else lb for lb in lbs]

clipscore_mean_ot = [clip_data['conditional_similarity'][
    (clip_data['strength'] == lb) & (clip_data['unconditional_similarity'] >= 0.0)
].sum() / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs]

clipscore_kas = [clip_data_kas['conditional_similarity'][
    (clip_data_kas['strength'] == lb) & (clip_data_kas['unconditional_similarity'] >= 0.0)
].sum()/ clip_data_kas[clip_data_kas['strength'] == 1].shape[0] for lb in lbs]

# --- First plot: line across all points ---
fig.add_trace(go.Scatter(
    x=x_lbs,
    y=clipscore_mean_ot,
    mode="lines+markers",
    line=dict(color='#4C9AFF', width=2),
    marker=dict(size=8),
    name="Mean-AcT"
))

# --- Clip Data (KAS) ---
fig.add_trace(go.Scatter(
    x=x_lbs,
    y=clipscore_kas,
    mode="lines+markers",
    line=dict(color='#FFB347', width=2),
    marker=dict(size=8),
    name="KAS"
))

# Layout
fig.update_layout(
    width=700,
    height=500,
    showlegend=True,
    title=dict(text="CLIPScore", x=0.5, xanchor="center"),
    xaxis=dict(
        domain=[0, 1.0],
        title="Strength λ",
        type="log",
        tickvals=[0.1, 1, 10],
        ticktext=["0.1", "1", "10"]
    ),
    yaxis=dict(title="ClipScores (style) (→)"),
    margin=dict(t=60, b=60)
)

fig.show()
output_path = Path(f"visualization/{model}/{style}/clipscore_{style}.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(output_path, width=700, height=500, scale=2)

In [ ]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplots: 1 row, 2 columns
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("0-shot", "CLIPScore"),
    shared_xaxes=True,
    horizontal_spacing=0.1,  # space between the two panels
)

# (Optional) ensure ordered x-axis
lbs = sorted(clip_data['strength'].unique())
x_lbs = [0.05 if lb == 0 else lb for lb in lbs]

# --- Zeroshot values ---
zeroshot_mean_ot = [
    clip_data[
        (clip_data['strength'] == lb) & (clip_data['conditional_zero_shot_score'] >= 0.49)
    ].shape[0] / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs
]
zeroshot_kas = [
    clip_data_kas[
        (clip_data_kas['strength'] == lb) & (clip_data_kas['conditional_zero_shot_score'] >= 0.49)
    ].shape[0] / clip_data_kas[clip_data_kas['strength'] == 1].shape[0] for lb in lbs
]

# --- CLIPScore values ---
clipscore_mean_ot = [
    clip_data['unconditional_similarity'][
        (clip_data['strength'] == lb) & (clip_data['unconditional_similarity'] >= 0.0)
    ].sum() / clip_data[clip_data['strength'] == 1].shape[0] for lb in lbs
]
clipscore_kas = [
    clip_data_kas['unconditional_similarity'][
        (clip_data_kas['strength'] == lb) & (clip_data_kas['unconditional_similarity'] >= 0.0)
    ].sum() / clip_data_kas[clip_data_kas['strength'] == 1].shape[0] for lb in lbs
]

# --- Top subplot (row 1): Zeroshot ---
fig.add_trace(go.Scatter(
    x=x_lbs, y=zeroshot_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT"
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=x_lbs, y=zeroshot_kas,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="KAS"  # fix label
), row=1, col=1)

# --- Right subplot (row 1, col 2): CLIPScore ---
fig.add_trace(go.Scatter(
    x=x_lbs, y=clipscore_mean_ot,
    mode="lines+markers",
    line=dict(color="#4C9AFF", width=2),
    marker=dict(size=8),
    name="Mean-AcT",
    showlegend=False  # avoid duplicate legends
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=x_lbs, y=clipscore_kas,
    mode="lines+markers",
    line=dict(color="#FFB347", width=2),
    marker=dict(size=8),
    name="KAS",
    showlegend=False
), row=1, col=2)

# Layout
fig.update_layout(
    width=900,
    height=250,             # taller for vertical stack
    showlegend=True,
    legend=dict(
        orientation="v",
        yanchor="middle",    # anchor vertically in the right margin
        y=0.5,
        xanchor="left",      # place outside the plotting panels
        x=1.02,
        bgcolor="rgba(255,255,255,0.6)",  # optional: semi-transparent background
        bordercolor="black",  # optional: border
        borderwidth=1,
        font=dict(size=15)
    ),
    margin=dict(t=55, b=40, r=180),
)

# Axis titles
fig.update_yaxes(title_text="0-shot (→)", row=1, col=1)
fig.update_yaxes(title_text="CLIPScore (→)", row=1, col=2)
fig.update_xaxes(
    title_text="λ",
    type="log",
    tickvals=[0.1, 1, 10],
    ticktext=["0.1", "1", "10"]
)

# Fonts / ticks
fig.update_layout(font=dict(size=30))
fig.update_annotations(font_size=28)
fig.update_xaxes(tickfont=dict(size=25), title_font=dict(size=30), tickformat=".2f")
fig.update_yaxes(tickfont=dict(size=25), title_font=dict(size=30), tickformat=".2f")

fig.show()

# Save (increase height for PDF export too)
output_path = Path(f"visualization/{model}/{style}/combined_{style}.pdf")
output_path.parent.mkdir(parents=True, exist_ok=True)
fig.write_image(
    output_path,
    width=1000, height=250, scale=2
)


In [ ]:
# data_dir = "results_42"

In [ ]:
# zero_shot_res = pd.read_csv(data_dir + '/' + 'evaluate_0shot/0shot_eval.csv', index_col=0)
# zero_shot_res[zero_shot_res['q0_llm_answer'] == "Yes"].shape[0] / zero_shot_res.shape[0]

In [ ]:
# mmlu_res = pd.read_pickle(data_dir+'/evaluate_eleuther/eleuther.pkl')

In [ ]:
# mmlu_res['results']['mmlu']['acc,none']

In [ ]:
# mistral_res = pd.read_csv(data_dir+'/evaluate_perplexity/model_perplexity.csv', index_col=0)
# mistral_res[mistral_res['strength']==1.0]['ppl_Mistral-7B-v0.1'].mean()